# Lesson 4: Probability & Statistics

*ML Academy — companion notebook*

Probability rules, Bayes' theorem, mean and variance, z-scores and the normal distribution.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-04.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 12. In Python

In [1]:
import numpy as np

scores = np.array([4, 6, 8, 10, 12])
print(scores.mean())           # 8.0
print(scores.var())            # 8.0   (divides by n)
print(scores.std())            # 2.828...
print(np.median([30, 35, 40, 45, 1000]))   # 40.0

z = (scores - scores.mean()) / scores.std()
print(z.round(2))              # [-1.41 -0.71  0.    0.71  1.41]

# Bayes: the medical test
p_d, sens, fpr = 0.01, 0.90, 0.05
p_pos = sens * p_d + fpr * (1 - p_d)
print(round(sens * p_d / p_pos, 3))        # 0.154

# Sample 100,000 heights and check the 68-95-99.7 rule
rng = np.random.default_rng(0)
h = rng.normal(170, 10, size=100_000)
print(np.mean(np.abs(h - 170) < 20))       # ≈ 0.95

8.0
8.0
2.8284271247461903
40.0
[-1.41 -0.71  0.    0.71  1.41]
0.154
0.9546


## 15. Going deeper: statistical inference for ML

In [2]:
import numpy as np
from scipy import stats

print("Binomial: P(8 or more of 10 visitors click | p 0.5):", round(stats.binom.sf(7, 10, 0.5), 4))
print("Poisson:  P(no support calls in an hour | 3 per hour):", round(stats.poisson.pmf(0, 3), 4))
print("Exponential: P(wait over 30 min | mean wait 20 min):", round(stats.expon.sf(30, scale=20), 4))

Binomial: P(8 or more of 10 visitors click | p 0.5): 0.0547
Poisson:  P(no support calls in an hour | 3 per hour): 0.0498
Exponential: P(wait over 30 min | mean wait 20 min): 0.2231


In [3]:
# Maximum likelihood for a coin: 7 heads in 10 flips
p_grid = np.linspace(0.01, 0.99, 99)
loglik = 7 * np.log(p_grid) + 3 * np.log(1 - p_grid)
print("p with the highest likelihood:", round(p_grid[np.argmax(loglik)], 2))

# MLE for a normal distribution divides by n, which underestimates the variance
rng = np.random.default_rng(0)
samples = rng.normal(0, 2, size=(100000, 5))            # 100,000 small samples, true variance 4
print("average variance, divide by n:    ", round(samples.var(axis=1, ddof=0).mean(), 3))
print("average variance, divide by n − 1:", round(samples.var(axis=1, ddof=1).mean(), 3))

p with the highest likelihood: 0.7
average variance, divide by n:     3.21
average variance, divide by n − 1: 4.012


In [4]:
# Bayesian updating: prior Beta(2, 2), then observe 7 heads and 3 tails
a, b = 2 + 7, 2 + 3
post = stats.beta(a, b)
print(f"posterior: Beta({a}, {b})")
print("posterior mean:", round(post.mean(), 3), " MAP (most likely value):", round((a - 1) / (a + b - 2), 3))
print("95% credible interval:", np.round(post.ppf([0.025, 0.975]), 3))

posterior: Beta(9, 5)
posterior mean: 0.643  MAP (most likely value): 0.667
95% credible interval: [0.386 0.861]


In [5]:
# Central limit theorem: averages of a skewed variable become bell-shaped
rng = np.random.default_rng(1)
for n in [1, 5, 30, 200]:
    means = rng.exponential(scale=1.0, size=(50000, n)).mean(axis=1)
    print(f"n {n:3d}: skewness {stats.skew(means):5.2f}   spread of the means {means.std():.3f}   1/sqrt(n) {1 / np.sqrt(n):.3f}")

n   1: skewness  1.98   spread of the means 0.997   1/sqrt(n) 1.000
n   5: skewness  0.90   spread of the means 0.445   1/sqrt(n) 0.447
n  30: skewness  0.37   spread of the means 0.182   1/sqrt(n) 0.183


n 200: skewness  0.15   spread of the means 0.071   1/sqrt(n) 0.071


In [6]:
# Bootstrap: a confidence interval for the median house price (simulated, lakh)
rng = np.random.default_rng(2)
prices = rng.lognormal(mean=4.2, sigma=0.5, size=60)
boot = [np.median(rng.choice(prices, size=len(prices), replace=True)) for _ in range(10000)]
print("sample median:", round(np.median(prices), 1))
print("95% bootstrap interval:", np.round(np.percentile(boot, [2.5, 97.5]), 1))

sample median: 72.6
95% bootstrap interval: [58.  78.7]


In [7]:
# A/B test: did the new checkout page raise the conversion rate?
conv = np.array([120, 156]); n = np.array([2400, 2400])
p1, p2 = conv / n
p_pool = conv.sum() / n.sum()
se = np.sqrt(p_pool * (1 - p_pool) * (1 / n[0] + 1 / n[1]))
z = (p2 - p1) / se
print(f"conversion: A {p1:.3f}, B {p2:.3f}   z {z:.2f}   two-sided p-value {2 * stats.norm.sf(abs(z)):.4f}")

conversion: A 0.050, B 0.065   z 2.23   two-sided p-value 0.0256


In [8]:
# Multiple testing: 20 features with no real effect, each tested at the 5% level, in 5,000 experiments
rng = np.random.default_rng(4)
groups = rng.normal(size=(5000, 20, 2, 30))              # two groups of 30, no true difference
p = stats.ttest_ind(groups[:, :, 0], groups[:, :, 1], axis=2).pvalue
print("share of single tests below 0.05:", round((p < 0.05).mean(), 3))
print("share of experiments with at least one 'significant' feature:", round((p < 0.05).any(axis=1).mean(), 3))
print("theory, 1 − 0.95^20:", round(1 - 0.95**20, 3))
print("with Bonferroni (p < 0.05/20):", round((p < 0.05 / 20).any(axis=1).mean(), 3))

share of single tests below 0.05: 0.05
share of experiments with at least one 'significant' feature: 0.643
theory, 1 − 0.95^20: 0.642
with Bonferroni (p < 0.05/20): 0.04


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [9]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
p = df["price_lakh"]
print("mean", round(p.mean(), 2), "std", round(p.std(ddof=0), 2))
print("z-score of house 12:", round((96 - p.mean()) / p.std(ddof=0), 2))
print("P(sold fast) =", df["sold_fast"].mean())
central = df[df["area"] == "Central"]
print("P(sold fast | Central) =", central["sold_fast"].mean())
print("P(Central | sold fast) =", round((df[df["sold_fast"] == 1]["area"] == "Central").mean(), 3))

mean 72.08 std 17.29
z-score of house 12: 1.38
P(sold fast) = 0.5
P(sold fast | Central) = 0.75
P(Central | sold fast) = 0.5


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [10]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: Standard deviation by hand

Compute the standard deviation (divide by n) without np.std.

In [11]:
import numpy as np
x = np.array([2, 4, 4, 4, 5, 5, 7, 9])
m = x.mean()
sd = None  # ← your code here
check(sd, 2.0, hint='Square root of the average squared deviation from the mean.')

❌ Not yet. Hint: Square root of the average squared deviation from the mean.


### Challenge 2: Bayes' rule

1% have a disease. The test catches 90% of cases and gives 5% false positives. Compute P(disease | positive).

In [12]:
p_d, p_pos_d, p_pos_h = 0.01, 0.90, 0.05
p_pos = p_pos_d * p_d + p_pos_h * (1 - p_d)
posterior = None  # ← your code here
check(posterior, 0.15384615384615385, hint="P(+|disease)·P(disease) / P(+). (It's only about 15%!)")

❌ Not yet. Hint: P(+|disease)·P(disease) / P(+). (It's only about 15%!)


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. Standard deviation by hand** — the missing line:

```python
sd = np.sqrt(np.mean((x - m) ** 2))
```

**2. Bayes' rule** — the missing line:

```python
posterior = p_pos_d * p_d / p_pos
```